# Testing: all four algorithms together

Runs **on your computer** after the 4 training notebooks have finished on Kaggle and `./kaggle.sh get` has downloaded
them into `downloads/`. It loads every trained model (best checkpoint + world model of each algorithm and seed) and the
training results, then:

1. **Loads** the 4 downloads and checks that every algorithm and seed is there.
2. **Training comparison**: all training results together: tables, smoothed learning curves, scores, trust in the
   world model, the world-model check.
3. **Test**: every model flies the same **50 new scenarios** (never used in training or for choosing the checkpoints);
   test tables and plots (returns, crashes, lap progress, conditions, flights, wind, MACURA's trust).
4. **Test videos**: all four algorithms on the same test flights, and each algorithm's 4 seeds.
5. **Final comparison**, and every model checked for **the simulator** (`mjpython simulate.py`).

How to run, from the repo folder: `jupyter notebook testing/test.ipynb` (then *Run All*), or without opening it:
`jupyter nbconvert --to notebook --execute --inplace testing/test.ipynb`. Everything is saved in `results/`.

## 1. Load the training results

In [ ]:
import os, sys, glob, json, zipfile

ROOT = os.path.abspath(".." if os.path.basename(os.getcwd()) == "testing" else ".")
os.chdir(ROOT)
sys.path.insert(0, ROOT)

DOWNLOADS = "downloads"          # ./kaggle.sh get puts each training notebook's output here
OUT_DIR   = "results"
EPISODES  = 50                   # new test scenarios per model (seeds 3000-3049)
VIDEOS    = True
EXPECTED  = {"macura": [0, 1, 2, 3], "mbpo": [0, 1, 2, 3], "m2ac": [0, 1, 2, 3], "sac": [0, 1, 2, 3]}

# a download that only has the notebook's results file <name>.zip (e.g. from the Kaggle website): unpack it
for z in sorted(glob.glob(f"{DOWNLOADS}/*/*.zip")):
    d = os.path.dirname(z)
    if not glob.glob(os.path.join(d, "**", "logs", "*_seed*.json"), recursive=True):
        zipfile.ZipFile(z).extractall(os.path.join(d, "runs")); print("unpacked", z)
# every downloaded result folder; if a notebook was downloaded more than once, the newest copy is used
found = sorted({os.path.dirname(os.path.dirname(p)) for p in glob.glob(f"{DOWNLOADS}/**/logs/*_seed*.json", recursive=True)
                if f"{os.sep}previous{os.sep}" not in p})
newest = {}
for f in found:
    info = os.path.join(f, "RUN_INFO.json")
    name = json.load(open(info)).get("notebook", f) if os.path.exists(info) else f
    if name not in newest or os.path.getmtime(f) > os.path.getmtime(newest[name]):
        newest[name] = f
FOLDERS = sorted(newest.values())
print("result folders used:"); [print("  ", f) for f in FOLDERS]
skipped = [f for f in found if f not in FOLDERS]
if skipped:
    print("older copies not used:", ", ".join(skipped))
first = json.load(open(glob.glob(f"{FOLDERS[0]}/logs/*.json")[0])) if FOLDERS else {}
TASK = first.get("setup", {}).get("task", "race3")
os.environ["MACURA_TASK"] = TASK
os.environ.setdefault("MUJOCO_GL", "disable")      # plots here; the videos are drawn in their own process
print("task:", TASK)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, Video, display
from src.config import config as cfg
from src.analysis import results as R, report, testing as T
from src.viz import figures as F, video as V, test_figures as TF

RUNS = R.load_runs(*FOLDERS)
WHERE = {(r["algo"], r["seed"]): os.path.relpath(r["_root"], ROOT) for r in RUNS}
rows = ["| algorithm | " + " | ".join(f"seed {s}" for s in range(4)) + " |", "|---|" + "---|" * 4]
for a in R.ALGOS:
    rows.append(f"| {R.NAMES[a]} | " + " | ".join(WHERE.get((a, s), "**missing**") for s in range(4)) + " |")
display(Markdown("### Which run comes from which download\n\n" + "\n".join(rows)))
missing = [f"{R.NAMES[a]} seed {s}" for a, ss in EXPECTED.items() for s in ss if (a, s) not in WHERE]
stopped = [f"{R.NAMES[r['algo']]} seed {r['seed']} (step {r['setup']['stopped_early_at']})" for r in RUNS
           if r.get("setup", {}).get("stopped_early_at")]
print("missing:", ", ".join(missing) if missing else "none")
print("stopped early by Kaggle's time limit:", ", ".join(stopped) if stopped else "none")
STEPS = R.run_length(RUNS[0])
print(f"{len(RUNS)} runs, {STEPS:,} steps each, task {TASK}")

## 2. Training: all four algorithms together
The scores fixed before the runs: **average return while learning** and **drones broken after the warm-up** (main),
then each best checkpoint's test on 30 fresh scenarios (return, **lap progress** = % of a lap per 10 s flight, crash
rate). With 4 seeds: IQM over seeds with a 95% bootstrap confidence interval.

In [ ]:
display(Markdown(R.summary_markdown(RUNS)))
display(Markdown("### Per seed\n\n" + R.per_seed_markdown(RUNS)))
GAP = [("avg_return", "avg return while learning", "{:+.0f}", 1), ("broken", "drones broken after warm-up", "{:+.0f}", -1),
       ("test_return", "final test return", "{:+.0f}", 1), ("test_success", "lap progress (test)", "{:+.0%}", 1),
       ("test_crash", "final test crash rate", "{:+.0%}", -1), ("last10_return", "return, last 10 evals", "{:+.0f}", 1)]
by = {(r["algo"], r["seed"]): R.run_scores(r) for r in RUNS}
TRAIN_GAPS = []
for other in ("mbpo", "m2ac", "sac"):
    seeds = [s for s in range(4) if ("macura", s) in by and (other, s) in by]
    if not seeds:
        continue
    t = [f"| MACURA - {R.NAMES[other]} | " + " | ".join(f"seed {s}" for s in seeds) + " | mean | MACURA better in |",
         "|---|" + "---|" * (len(seeds) + 2)]
    for k, lab, fmt, sign in GAP:
        d = [by[("macura", s)][k] - by[(other, s)][k] for s in seeds]
        t.append(f"| {lab} | " + " | ".join(fmt.format(x) for x in d) + f" | {fmt.format(np.mean(d))} | "
                 f"{sum(sign * x > 0 for x in d)} of {len(d)} |")
    TRAIN_GAPS.append(f"#### Training: MACURA vs {R.NAMES[other]} (same seed = same scenarios, wind and gusts)\n\n" + "\n".join(t))
    display(Markdown(TRAIN_GAPS[-1]))

### Learning curves (smoothed) and scores
Rolling mean over 7 evaluations (raw values faint), shaded = 95% CI over the seeds; dotted lines = the hand-written
autopilot.

In [ ]:
REFS = R.autopilot_reference(cfg.ENV, seed_base=100, episodes=20, cache=f"{OUT_DIR}/autopilot_reference.json")
F.plot_learning_curves(RUNS, REFS, f"{OUT_DIR}/plots/learning_curves_smooth.png", window=7); plt.show()
for a in R.ALGOS:
    rs = [r for r in RUNS if r["algo"] == a]
    if rs:
        F.plot_algo(rs, REFS, f"{OUT_DIR}/plots/{a}_per_seed.png", window=7); plt.show()
F.plot_summary_table(RUNS, f"{OUT_DIR}/plots/summary_table.png"); plt.show()
F.plot_scores(RUNS, f"{OUT_DIR}/plots/scores.png"); plt.show()

### The world model and trust (training)
How much of its imagination each method trusts, MACURA's threshold kappa over training, how much imagined data above
MACURA's threshold MBPO and M2AC trained on, updates per real step (fairness check), the chute, and the world-model
check (the best MACURA and MBPO models on real flights: disagreement vs real error).

In [ ]:
print(R.trust_summary(RUNS))
UTD = ["| algorithm | " + " | ".join(f"seed {s}" for s in range(4)) + " |", "|---|" + "---|" * 4]
for a in R.ALGOS:
    cells = []
    for s in range(4):
        r = next((x for x in RUNS if x["algo"] == a and x["seed"] == s), None)
        cells.append("-" if r is None else (f"{np.mean(r['utd']):.1f}" if r.get("utd") else "1"))
    UTD.append(f"| {R.NAMES[a]} | " + " | ".join(cells) + " |")
display(Markdown("#### Updates per real step (mean)\n\n" + "\n".join(UTD)))
CHUTE = ["| algorithm | seed | fastest descent, 2nd half (m/s) | losing lift, 2nd half | test: losing lift |", "|---|---|---|---|---|"]
for r in RUNS:
    late = np.asarray(r["steps"]) >= STEPS / 2
    sink, lift = np.asarray(r["eval_max_sink"])[late], np.asarray(r["eval_liftloss"])[late]
    CHUTE.append(f"| {R.NAMES[r['algo']]} | {r['seed']} | {sink.max() if len(sink) else float('nan'):.2f} | "
                 f"{100 * lift.mean() if len(lift) else float('nan'):.1f}% | {100 * r['final_eval'].get('eval_liftloss', 0):.1f}% |")
display(Markdown("#### The chute (lift loss starts at 1.2 m/s of sink)\n\n" + "\n".join(CHUTE)))
MADE = report.build(RUNS, cfg.CFG, OUT_DIR, check=True, videos=False, close=True)
for name in ("model_trust", "kappa", "untrusted_data", "model_check", "crash_by_payload"):
    if MADE.get(name):
        print(name); display(Image(MADE[name]))

## 3. Test on new scenarios
Every model (the best checkpoint of each algorithm and seed) flies the same **50 new scenarios** (seeds 3000-3049:
new starts, packages, winds and gusts; never used in training, for choosing the checkpoint, or in the training
notebooks' test). The first 3 flights of each model are recorded step by step for the plots. Several models are tested
at the same time; a few minutes.

In [ ]:
TESTS = T.test_all(RUNS, cfg.CFG, episodes=EPISODES)
TABLES = T.test_tables(TESTS)
AP_TEST = R.autopilot_reference(cfg.ENV, seed_base=T.TEST_SEED_BASE, episodes=EPISODES,
                                cache=f"{OUT_DIR}/autopilot_reference_new_test.json")
display(Markdown("### Test results\n\n" + TABLES["summary"]))
print("hand-written autopilot on the same scenarios:", ", ".join(
    f"{k}: return {v['return']:.0f}, lap {100 * min(max(v['laps'], 0), 1):.0f}%, crash {100 * v['crash']:.0f}%" for k, v in AP_TEST.items()))
display(Markdown("### Per seed\n\n" + TABLES["per_seed"]))
for t in TABLES["paired"]:
    display(Markdown(t))

In [ ]:
TF.plot_test_scores(TESTS, f"{OUT_DIR}/plots/test_scores.png"); plt.show()
TF.plot_test_seeds(TESTS, f"{OUT_DIR}/plots/test_seeds.png"); plt.show()
TF.plot_test_conditions(TESTS, f"{OUT_DIR}/plots/test_conditions.png"); plt.show()
for i in range(2):
    TF.plot_test_flights(TESTS, cfg.CFG, trace=i, save_path=f"{OUT_DIR}/plots/test_flight_{i}.png"); plt.show()
TF.plot_test_noise(TESTS, save_path=f"{OUT_DIR}/plots/test_wind.png"); plt.show()
fig = TF.plot_test_trust(TESTS, f"{OUT_DIR}/plots/test_trust.png")
if fig is not None:
    plt.show()

## 4. Test videos
The best seed of each algorithm (best on the selection scenarios, never picked on test scenarios) flies the **same
three new test scenarios** side by side (same start, package, wind and gusts); then each algorithm's four seeds on the
same scenario; then a 30 s MACURA vs MBPO flight. The trail turns red while a drone loses lift; MACURA's panel shows its
world model's verdict at every step (green = its models agree, orange = they disagree).

In [ ]:
def trust_of(r):
    return {"ensemble": R.ensemble_checkpoint(r), "kappa": R.kappa_at(r)} if r["algo"] == "macura" and R.ensemble_checkpoint(r) else None

if VIDEOS:
    best = {R.NAMES[a]: R.best_run(rs) for a, rs in R.by_algo(RUNS).items()}
    print("best seed per algorithm:", {k: r["seed"] for k, r in best.items()})
    trust = {k: trust_of(r) for k, r in best.items() if trust_of(r)}
    p = V.record_race(cfg.CFG, {k: R.checkpoint(r) for k, r in best.items()}, f"{OUT_DIR}/videos/test_all_algorithms.mp4",
                      seeds=(T.TEST_SEED_BASE, T.TEST_SEED_BASE + 1, T.TEST_SEED_BASE + 2), trust=trust, frame_step=3, fps=17)
    if p:
        display(Video(p, embed=True, width=960))
    for a, rs in R.by_algo(RUNS).items():
        pilots = {f"{R.NAMES[a]} seed {r['seed']}": R.checkpoint(r) for r in rs}
        tr = {f"{R.NAMES[a]} seed {r['seed']}": trust_of(r) for r in rs if trust_of(r)}
        p = V.record_race(cfg.CFG, pilots, f"{OUT_DIR}/videos/test_{a}_all_seeds.mp4", seeds=(T.TEST_SEED_BASE + 3,),
                          trust=tr, frame_step=3, fps=17)
        print(R.NAMES[a], "- all seeds:", p)
        if p:
            display(Video(p, embed=True, width=960))
    duo = {k: R.checkpoint(r) for k, r in best.items() if k in ("MACURA", "MBPO")}
    if len(duo) == 2:
        p = V.record_race(cfg.CFG, duo, f"{OUT_DIR}/videos/test_long_flight.mp4", seeds=(T.TEST_SEED_BASE + 4,),
                          trust=trust, frame_step=3, fps=17, max_steps=1500)
        if p:
            display(Video(p, embed=True, width=960))

## 5. Final comparison
Training (average return while learning, drones broken) and test on the new scenarios (return, lap progress, crash
rate), per algorithm: IQM over the 4 seeds.

In [ ]:
train = R.summary(RUNS)
test = {}
for res in TESTS:
    test.setdefault(res["algo"], []).append(T.seed_scores(res))
final = ["| algorithm | avg return while learning | drones broken | test return (new scenarios) | test lap progress | "
         "test crash rate |", "|---|---|---|---|---|---|"]
for a in [x for x in R.ALGOS if x in train]:
    ts = test.get(a, [])
    final.append(f"| {R.NAMES[a]} | {train[a]['avg_return'][0]:.0f} | {train[a]['broken'][0]:.0f} | "
                 f"{R.iqm([s['return'] for s in ts]):.0f} | {100 * R.iqm([s['lap'] for s in ts]):.0f}% | "
                 f"{100 * R.iqm([s['crash'] for s in ts]):.0f}% |")
FINAL = "\n".join(final)
display(Markdown("### Final comparison (IQM over seeds)\n\n" + FINAL))
parts = ["# Final results", FINAL, "## Training", R.summary_markdown(RUNS), R.per_seed_markdown(RUNS), *TRAIN_GAPS,
         "## Updates per real step", "\n".join(UTD), "## The chute", "\n".join(CHUTE),
         f"## Test on {EPISODES} new scenarios", TABLES["summary"], TABLES["per_seed"], *TABLES["paired"],
         "## Which run comes from which download", "\n".join(rows)]
open(f"{OUT_DIR}/summary.md", "w").write("\n\n".join(parts) + "\n")
json.dump([{"algo": r["algo"], "seed": r["seed"], "episodes": r["episodes"]} for r in TESTS],
          open(f"{OUT_DIR}/test_results.json", "w"), default=float)
print("saved:", os.path.abspath(OUT_DIR))

## 6. Ready for the simulator
Every trained model is loaded once more to make sure it opens on this computer (the checkpoints are saved on Kaggle
with a newer numpy; the loader handles it). Then run the simulator in a terminal (macOS: `mjpython`, not `python`):

In [ ]:
from src.envs import drone_env
from src.algorithms.sac import load_agent
env, od, ad = drone_env.make_env(cfg.ENV, seed=0)
ok = []
for r in RUNS:
    try:
        load_agent(R.checkpoint(r), od, ad, cfg.SAC, "cpu"); ok.append(f"{r['algo']} s{r['seed']}")
    except Exception as e:
        print("could not load", r["algo"], r["seed"], e)
env.close()
print(f"{len(ok)} of {len(RUNS)} models load:", ", ".join(ok))
print()
print("in a terminal, from the repo folder:")
print("  mjpython simulate.py                              # best seed of each algorithm, test scenarios 1000-1004")
print("  mjpython simulate.py --scenarios 3000 3001 3002   # the new test scenarios used above")
print("  mjpython simulate.py --algos macura mbpo --seconds 30 --follow MACURA")
print("  mjpython simulate.py --seed 2                     # seed 2 of every algorithm")